<a href="https://colab.research.google.com/github/HisameOgasahara/manga2text_tmp/blob/main/test2_taggr_OCR_colab_secret.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

https://github.com/sorryhyun/anime_tools 참조

In [1]:
import subprocess
from pathlib import Path

REPO = Path("/content/anime_tools")
ENV = Path("/content/anime_env")
PY = ENV / "bin/python"
UV = "/usr/local/bin/uv"

print("=== GPU CHECK ===")

gpu = subprocess.run(
    ["nvidia-smi"],
    text=True,
    capture_output=True,
)

print(gpu.stdout)

if gpu.returncode != 0:
    raise RuntimeError(
        "GPU 런타임이 아닙니다. "
        "Colab → 런타임 → 런타임 유형 변경 → T4 GPU 선택 후 다시 실행하세요."
    )

print("\n=== CLEAN ===")

subprocess.run(
    ["rm", "-rf", str(REPO), str(ENV)],
    check=True,
)

print("\n=== INSTALL UV ===")

subprocess.run(
    ["bash", "-lc", "curl -LsSf https://astral.sh/uv/install.sh | sh"],
    check=True,
)

print("\n=== INSTALL PYTHON 3.13 ===")

subprocess.run(
    [UV, "python", "install", "3.13"],
    check=True,
)

print("\n=== CREATE VENV ===")

subprocess.run(
    [
        UV,
        "venv",
        "--python",
        "3.13",
        str(ENV),
    ],
    check=True,
)

print("\n=== CLONE REPO ===")

subprocess.run(
    [
        "git",
        "clone",
        "https://github.com/sorryhyun/anime_tools.git",
        str(REPO),
    ],
    check=True,
)

print("\n=== INSTALL DEPENDENCIES ===")

deps = [
    "torch>=2.12",
    "torchvision",
    "numpy>=2.0",
    "Pillow",
    "pyyaml",
    "safetensors>=0.5",
    "huggingface-hub>=0.30",
    "tqdm",
    "rich>=13",
    "timm>=1.0",
    "einops",
    "transformers>=5.16",
    "sentencepiece",
    "peft>=0.20",
    "opencv-python>=4.11",
    "accelerate",
]

subprocess.run(
    [
        UV,
        "pip",
        "install",
        "--python",
        str(PY),
        *deps,
    ],
    check=True,
)

print("\n=== INSTALL ANIME_TOOLS WITHOUT EXTRA DEPS ===")

subprocess.run(
    [
        UV,
        "pip",
        "install",
        "--python",
        str(PY),
        "--no-deps",
        "-e",
        str(REPO),
    ],
    check=True,
)

print("\n=== VERIFY ===")

verify_code = r"""
import sys
import torch
import PIL
import anime_tools

print("Python :", sys.version)
print("Torch  :", torch.__version__)
print("Pillow :", PIL.__version__)
print("CUDA   :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU    :", torch.cuda.get_device_name(0))
    print(
        "VRAM   :",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        ),
        "GB"
    )

assert torch.cuda.is_available()
print("anime_tools import OK")
"""

subprocess.run(
    [
        str(PY),
        "-c",
        verify_code,
    ],
    check=True,
)

print("\n=== SETUP COMPLETE ===")

=== GPU CHECK ===
Tue Sep 22 01:07:45 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------

In [2]:
#@title Hugging Face Secret 확인
# Colab 왼쪽 🔑 Secrets에 이름을 정확히 `HF_TOKEN`으로 추가하고
# "Notebook access"를 켠 뒤 이 셀을 실행하세요.

import os
import subprocess
from google.colab import userdata

PY = "/content/anime_env/bin/python"

print("=== LOAD COLAB SECRET ===")

try:
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception as exc:
    raise RuntimeError(
        "Colab Secrets에서 `HF_TOKEN`을 찾지 못했습니다. "
        "왼쪽 🔑 Secrets에 추가하고 Notebook access를 켜세요."
    ) from exc

if not HF_TOKEN or not HF_TOKEN.startswith("hf_"):
    raise RuntimeError("`HF_TOKEN` 값이 비어 있거나 Hugging Face 토큰 형식이 아닙니다.")

# 현재 Colab 커널 환경에만 넣습니다.
# 출력하거나 파일에 저장하지 않습니다.
os.environ["HF_TOKEN"] = HF_TOKEN

check_code = r"""
import os
from huggingface_hub import HfApi, hf_hub_url, get_hf_file_metadata

token = os.environ["HF_TOKEN"]
api = HfApi()

print("=== TOKEN VALIDITY ===")

me = api.whoami(token=token)
name = me.get("name") or me.get("fullname") or "(unknown)"
print("HF account:", name)
print("Token: VALID")

checks = [
    (
        "Anima Tagger backbone",
        "animetimm/caformer_b36.dbv4-full",
        "model.safetensors",
    ),
    (
        "AnimeText OCR detector",
        "deepghs/AnimeText_yolo",
        "yolo12l_animetext/model.pt",
    ),
]

print()
print("=== GATED ACCESS CHECK ===")

failed = []

for label, repo_id, filename in checks:
    try:
        url = hf_hub_url(
            repo_id=repo_id,
            filename=filename,
        )

        get_hf_file_metadata(
            url,
            token=token,
        )

        print(f"[PASS] {label}")
        print(f"       {repo_id}/{filename}")

    except Exception as exc:
        failed.append((label, repo_id, type(exc).__name__, str(exc)))
        print(f"[FAIL] {label}")
        print(f"       {repo_id}")
        print(f"       {type(exc).__name__}: {exc}")

if failed:
    print()
    print("=== ACTION REQUIRED ===")
    for label, repo_id, _, _ in failed:
        print(f"- {label}: https://huggingface.co/{repo_id}")
    raise SystemExit(
        "하나 이상의 gated model 접근권한이 없습니다. "
        "위 모델 페이지에서 접근 승인을 받은 뒤 이 셀을 다시 실행하세요."
    )

print()
print("=== HF READY ===")
"""

env = os.environ.copy()

result = subprocess.run(
    [PY, "-c", check_code],
    env=env,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

print(result.stdout)

if result.returncode != 0:
    raise RuntimeError(
        "HF 토큰 또는 gated model 접근권한 검사에 실패했습니다."
    )


=== LOAD COLAB SECRET ===
=== TOKEN VALIDITY ===
HF account: HisameOgasahara
Token: VALID

=== GATED ACCESS CHECK ===
[PASS] Anima Tagger backbone
       animetimm/caformer_b36.dbv4-full/model.safetensors
[PASS] AnimeText OCR detector
       deepghs/AnimeText_yolo/yolo12l_animetext/model.pt

=== HF READY ===



In [3]:
from google.colab import files
from pathlib import Path

uploaded = files.upload()

filename = next(iter(uploaded))

IMAGE_PATH = Path("/content") / filename

Path("/content/image_path.txt").write_text(
    str(IMAGE_PATH),
    encoding="utf-8",
)

print("IMAGE_PATH =", IMAGE_PATH)

Saving 1.jpg to 1.jpg
IMAGE_PATH = /content/1.jpg


In [4]:
#@title Anima Tagger
import os
import subprocess
from pathlib import Path

PY = "/content/anime_env/bin/python"
SCRIPT = Path("/content/run_tagger.py")

code = r'''
from pathlib import Path
from PIL import Image

from anime_tools.tagger import AnimaTagger
from anime_tools.tagger.fetch import (
    ensure_tagger_checkpoint,
    ensure_tagger_backbone,
)

# --------------------------------------------------
# 이미지 경로
# --------------------------------------------------

image_path = Path(
    Path("/content/image_path.txt")
    .read_text(encoding="utf-8")
    .strip()
)

# --------------------------------------------------
# Tagger checkpoint
# --------------------------------------------------

ckpt_dir = Path(
    "/content/anime_tools/models/captioners/anima-tagger-dbv4"
)

print("=== FETCH CHECKPOINT ===", flush=True)

ckpt_dir = ensure_tagger_checkpoint(
    ckpt_dir
)

print("=== FETCH BACKBONE ===", flush=True)

ensure_tagger_backbone(
    ckpt_dir
)

# --------------------------------------------------
# 모델 로드
# --------------------------------------------------

print("=== LOAD TAGGER ===", flush=True)

tagger = AnimaTagger(
    ckpt_dir,
    device="cuda",
)

# --------------------------------------------------
# 이미지 로드
# --------------------------------------------------

image = Image.open(
    image_path
).convert("RGB")

print("image:", image_path)
print("size :", image.size)

# --------------------------------------------------
# 추론
# --------------------------------------------------

print("=== TAGGING ===", flush=True)

result = tagger.predict(
    image
)

caption = tagger._caption_of(
    result,
    min_confidence=0.0,
)

# --------------------------------------------------
# 최종 caption
# --------------------------------------------------

print()
print("========== CAPTION ==========")
print(caption)

# --------------------------------------------------
# Rating
# --------------------------------------------------

print()
print("========== RATING ==========")
print(result["rating"])

# --------------------------------------------------
# 캐릭터만
# --------------------------------------------------

print()
print("========== CHARACTERS ==========")

characters = []

for tag, score in result["kept"].items():
    if tagger._cat_of.get(tag) == "character":
        characters.append(
            (tag, score)
        )

characters.sort(
    key=lambda item: item[1],
    reverse=True,
)

if characters:
    for tag, score in characters:
        print(
            f"{score:.3f}  {tag}"
        )
else:
    print(
        "No character tag survived post-processing."
    )

# --------------------------------------------------
# Copyright / 작품명
# --------------------------------------------------

print()
print("========== COPYRIGHT / SERIES ==========")

copyrights = []

for tag, score in result["kept"].items():
    if tagger._cat_of.get(tag) == "copyright":
        copyrights.append(
            (tag, score)
        )

copyrights.sort(
    key=lambda item: item[1],
    reverse=True,
)

if copyrights:
    for tag, score in copyrights:
        print(
            f"{score:.3f}  {tag}"
        )
else:
    print(
        "No copyright tag."
    )

# --------------------------------------------------
# Artist
# --------------------------------------------------

print()
print("========== ARTIST ==========")

artists = []

for tag, score in result["kept"].items():
    if tagger._cat_of.get(tag) == "artist":
        artists.append(
            (tag, score)
        )

artists.sort(
    key=lambda item: item[1],
    reverse=True,
)

if artists:
    for tag, score in artists:
        print(
            f"{score:.3f}  {tag}"
        )
else:
    print(
        "No artist tag."
    )

# --------------------------------------------------
# 최종적으로 살아남은 모든 tag
# --------------------------------------------------

print()
print("========== ALL KEPT TAGS ==========")

for tag, score in sorted(
    result["kept"].items(),
    key=lambda item: item[1],
    reverse=True,
):
    category = tagger._cat_of.get(
        tag,
        "unknown",
    )

    print(
        f"{score:.3f}  "
        f"[{category}]  "
        f"{tag}"
    )

# --------------------------------------------------
# 그룹 추론
# --------------------------------------------------

if "groups" in result:

    print()
    print("========== GROUPS ==========")

    for name, value in result["groups"].items():
        print(
            f"{name}: {value}"
        )

# --------------------------------------------------
# 캐릭터 raw candidate 확인
#
# character_floor 때문에 최종 caption에서
# 잘린 캐릭터도 확인 가능
# --------------------------------------------------

print()
print("========== RAW CHARACTER SCORES ==========")

raw_characters = []

for tag, score in result["scores"].items():

    if tagger._cat_of.get(tag) != "character":
        continue

    raw_characters.append(
        (
            tag,
            score,
            result["thresholds"].get(tag),
        )
    )

raw_characters.sort(
    key=lambda item: item[1],
    reverse=True,
)

for tag, score, threshold in raw_characters[:20]:

    survived = tag in result["kept"]

    print(
        f"{score:.3f}  "
        f"threshold={threshold:.3f}  "
        f"survived={survived}  "
        f"{tag}"
    )
'''

SCRIPT.write_text(
    code,
    encoding="utf-8",
)

env = os.environ.copy()

if not env.get("HF_TOKEN"):
    raise RuntimeError("먼저 `Hugging Face Secret 확인` 셀을 실행하세요.")

result = subprocess.run(
    [
        PY,
        str(SCRIPT),
    ],
    env=env,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

print(result.stdout)

if result.returncode != 0:
    raise RuntimeError(
        f"Tagger failed: exit code {result.returncode}"
    )


=== FETCH CHECKPOINT ===
=== FETCH BACKBONE ===
=== LOAD TAGGER ===
image: /content/1.jpg
size : (900, 1200)
=== TAGGING ===

========== CAPTION ==========
safe, 1boy, 1girl, blue archive, :d, ^^^, ^ ^, black gloves, black jacket, black shoes, blush, blue necktie, closed eyes, comic, collared shirt, flying sweatdrops, halo, hair between eyes, long hair, long sleeves, necktie, open clothes, open mouth, parted bangs, nose blush, open jacket, purple hair, puffy sleeves, puffy long sleeves, smile, sweat, white shirt, two side up, white jacket, v-shaped eyebrows

========== RATING ==========
safe

========== CHARACTERS ==========
No character tag survived post-processing.

========== COPYRIGHT / SERIES ==========
1.000  blue archive

========== ARTIST ==========
No artist tag.

========== ALL KEPT TAGS ==========
1.000  [copyright]  blue archive
0.999  [count]  1girl
0.967  [general]  blush
0.966  [general]  comic
0.941  [general]  jacket
0.928  [general]  shirt
0.927  [count]  1boy
0.910  

In [5]:
#@title OCR Test
import os
import subprocess
from pathlib import Path

PY = "/content/anime_env/bin/python"
SCRIPT = Path("/content/run_ocr.py")

code = r'''
import traceback
from pathlib import Path

try:
    import torch

    from anime_tools.ocr import load_ocr
    from anime_tools.ocr.reread import RereadEngine
    from anime_tools.ocr.sfx import SfxReader

    image_path = Path(
        Path("/content/image_path.txt")
        .read_text(encoding="utf-8")
        .strip()
    )

    print("=== GPU ===", flush=True)
    print("CUDA:", torch.cuda.is_available(), flush=True)

    if not torch.cuda.is_available():
        raise RuntimeError("CUDA unavailable")

    print(
        "GPU:",
        torch.cuda.get_device_name(0),
        flush=True,
    )

    print(
        "VRAM:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2,
        ),
        "GB",
        flush=True,
    )

    print("\n=== IMAGE ===", flush=True)
    print(image_path, flush=True)

    # --------------------------------------------------
    # AnimeText detector
    # --------------------------------------------------

    print(
        "\n=== LOAD ANIMETEXT DETECTOR ===",
        flush=True,
    )

    detector = load_ocr(
        device="cuda",
        min_box_px=12,
        max_boxes=64,
        det_conf=0.25,
    )

    print(
        "AnimeText detector loaded.",
        flush=True,
    )

    # --------------------------------------------------
    # Manga VL reader
    # --------------------------------------------------

    print(
        "\n=== LOAD MANGA VL READER ===",
        flush=True,
    )

    reader = SfxReader.load(
        device="cuda",
        batch_size=1,
    )

    print(
        "Manga VL reader loaded.",
        flush=True,
    )

    # --------------------------------------------------
    # OCR pipeline
    # --------------------------------------------------

    engine = RereadEngine(
        engine=detector,
        read_boxes=reader.read_boxes_scored,
        resized_dir=image_path.parent,
        masks=None,
        min_chars=1,
        skip_en=False,
        min_det=0.25,
        min_score=0.30,
        strip_symbols=False,
    )

    print(
        "\n=== RUN OCR ===",
        flush=True,
    )

    lines = engine.read(image_path)

    print(
        "\n========== OCR RESULT ==========",
        flush=True,
    )

    if not lines:
        print(
            "텍스트를 찾지 못했습니다.",
            flush=True,
        )
    else:
        for line in lines:
            print(
                f"[{line.seq}] {line.text}",
                flush=True,
            )
            print(
                f"  box      : {line.box}",
                flush=True,
            )
            print(
                f"  detector : {line.det:.3f}",
                flush=True,
            )
            print(
                f"  reader   : {line.score:.3f}",
                flush=True,
            )
            print("", flush=True)

except Exception:
    print(
        "\n========== OCR ERROR ==========\n",
        flush=True,
    )
    traceback.print_exc()
    raise
'''

SCRIPT.write_text(
    code,
    encoding="utf-8",
)

env = os.environ.copy()

if not env.get("HF_TOKEN"):
    raise RuntimeError("먼저 `Hugging Face Secret 확인` 셀을 실행하세요.")

result = subprocess.run(
    [PY, str(SCRIPT)],
    env=env,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)

print(result.stdout)

if result.stderr:
    print(
        "\n========== STDERR ==========\n"
    )
    print(result.stderr)

print(
    "\nexit code:",
    result.returncode,
)


=== GPU ===
CUDA: True
GPU: Tesla T4
VRAM: 14.56 GB

=== IMAGE ===
/content/1.jpg

=== LOAD ANIMETEXT DETECTOR ===
  deepghs/AnimeText_yolo/yolo12l_animetext/model.pt
    ok  /content/models/animetext/model.pt  (54 MB)
  deepghs/AnimeText_yolo/yolo12l_animetext/threshold.json
    ok  /content/models/animetext/threshold.json  (0 KB)
AnimeText detector loaded.

=== LOAD MANGA VL READER ===
  PaddlePaddle/PaddleOCR-VL-1.6/config.json
    ok  /content/models/paddleocr_vl_1.6/config.json  (2 KB)
  PaddlePaddle/PaddleOCR-VL-1.6/generation_config.json
    ok  /content/models/paddleocr_vl_1.6/generation_config.json  (0 KB)
  PaddlePaddle/PaddleOCR-VL-1.6/model.safetensors
    ok  /content/models/paddleocr_vl_1.6/model.safetensors  (1,917 MB)
  PaddlePaddle/PaddleOCR-VL-1.6/added_tokens.json
    ok  /content/models/paddleocr_vl_1.6/added_tokens.json  (25 KB)
  PaddlePaddle/PaddleOCR-VL-1.6/special_tokens_map.json
    ok  /content/models/paddleocr_vl_1.6/special_tokens_map.json  (1 KB)
  PaddleP

In [6]:
#@title OCR Reading Order

import re
from dataclasses import dataclass


# --------------------------------------------------
# 이전 OCR 셀의 출력 가져오기
# --------------------------------------------------

ocr_output = result.stdout

if not ocr_output:
    raise RuntimeError(
        "이전 OCR 셀의 result.stdout이 없습니다."
    )


# --------------------------------------------------
# OCR 결과 파싱
# --------------------------------------------------

@dataclass
class OCRLine:
    text: str
    box: tuple[int, int, int, int]

    @property
    def width(self):
        return self.box[2] - self.box[0]

    @property
    def height(self):
        return self.box[3] - self.box[1]


pattern = re.compile(
    r"\[(\d+)\]\s+(.*?)\n"
    r"\s*box\s*:\s*\((\d+),\s*(\d+),\s*(\d+),\s*(\d+)\)",
    re.MULTILINE,
)

lines = []

for match in pattern.finditer(ocr_output):

    _, text, x0, y0, x1, y1 = match.groups()

    lines.append(
        OCRLine(
            text=text.strip(),
            box=(
                int(x0),
                int(y0),
                int(x1),
                int(y1),
            ),
        )
    )

if not lines:
    raise RuntimeError(
        "OCR 결과에서 box/text를 찾지 못했습니다."
    )


# --------------------------------------------------
# sorryhyun reading_order와 같은 핵심 로직
#
# 세로 박스가 절반 이상:
#     오른쪽 → 왼쪽
#     같은 column에서는 위 → 아래
#
# 그 외:
#     위 → 아래
#     같은 row에서는 왼쪽 → 오른쪽
# --------------------------------------------------

VERTICAL_RATIO = 1.5


def is_vertical(line):

    return (
        line.height
        >= VERTICAL_RATIO * max(line.width, 1)
    )


vertical_count = sum(
    is_vertical(line)
    for line in lines
)


if vertical_count * 2 >= len(lines):

    mode = "VERTICAL"

    widths = sorted(
        max(1, line.width)
        for line in lines
    )

    median_width = widths[len(widths) // 2]

    band = max(
        1,
        median_width // 2,
    )

    ordered = sorted(
        lines,
        key=lambda line: (
            -(line.box[2] // band),
            line.box[1],
        ),
    )

else:

    mode = "HORIZONTAL"

    heights = sorted(
        max(1, line.height)
        for line in lines
    )

    median_height = heights[len(heights) // 2]

    band = max(
        1,
        median_height // 2,
    )

    ordered = sorted(
        lines,
        key=lambda line: (
            line.box[1] // band,
            line.box[0],
        ),
    )


# --------------------------------------------------
# 출력
# --------------------------------------------------

print("========== READING ORDER ==========")
print("mode:", mode)
print()

for index, line in enumerate(
    ordered,
    start=1,
):

    print(
        f"[{index}] {line.text}"
    )

    print(
        f"    box={line.box}"
    )


print()
print("========== JOINED TEXT ==========")
print()

for line in ordered:
    print(line.text)

========== READING ORDER ==========
mode: VERTICAL

[1] ようこそ先生!セミナー会計のユウカです!
    box=(689, 64, 794, 246)
[2] じゃ、じゃあユウカのさ
    box=(690, 362, 782, 549)
[3] なななな…!! セクハラです先生!
    box=(707, 628, 790, 860)
[4] もう…仕方ないですねサ、サイズは上からごにょごにょ…
    box=(725, 955, 808, 1114)
[5] ヒソ…
    box=(661, 1113, 728, 1157)
[6] テレ
    box=(584, 527, 636, 577)
[7] 分からないことは私に何でも聞いて下さい!
    box=(316, 45, 387, 261)
[8] テレ
    box=(338, 534, 379, 574)
[9] ギョーン!
    box=(257, 928, 406, 1001)
[10] 何でも!?
    box=(167, 30, 223, 154)
[11] スリーサイズとか聞いてもいいの?
    box=(124, 345, 213, 567)
[12] ああああッッ!!なんやかんやで教えてくれるタイプの子だぉ!
    box=(36, 912, 220, 1174)
[13] だ、だよね!ごめんごめん
    box=(66, 655, 158, 838)

========== JOINED TEXT ==========

ようこそ先生!セミナー会計のユウカです!
じゃ、じゃあユウカのさ
なななな…!! セクハラです先生!
もう…仕方ないですねサ、サイズは上からごにょごにょ…
ヒソ…
テレ
分からないことは私に何でも聞いて下さい!
テレ
ギョーン!
何でも!?
スリーサイズとか聞いてもいいの?
ああああッッ!!なんやかんやで教えてくれるタイプの子だぉ!
だ、だよね!ごめんごめん
